# Chest X-Ray Pneumonia Classifier — ResNet18 Transfer Learning

Fine-tunes an ImageNet-pretrained **ResNet18** for binary classification (`Normal` vs `Pneumonia`).

**Outputs**
- Best checkpoint → `backend/model_weights/best_model.pth`
- Metrics → accuracy, precision, recall, F1, AUC on the test set

**Expected data layout**
```
data/
  train/{Normal,Pneumonia}/*
  val/{Normal,Pneumonia}/*      # optional — carved from train if missing
  test/{Normal,Pneumonia}/*
```

## 1. Imports & configuration

In [ ]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import models, transforms
from tqdm.auto import tqdm

# Paths (notebook lives in notebooks/)
REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
WEIGHTS_DIR = REPO_ROOT / "backend" / "model_weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = WEIGHTS_DIR / "best_model.pth"
METRICS_PATH = WEIGHTS_DIR / "metrics_report.json"

# Hyperparameters
CLASS_NAMES = ["Normal", "Pneumonia"]
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2  # used when data/val is missing
NUM_WORKERS = 0
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Data dir: {DATA_DIR}")
print(f"Checkpoint: {BEST_MODEL_PATH}")

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

## 2. Transforms (augmentation) & dataset

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


class ChestXrayFolder(Dataset):
    """Loads images from root/{Normal,Pneumonia}/ or root/{split}/{Normal,Pneumonia}/."""

    def __init__(self, root: Path, transform=None) -> None:
        self.root = Path(root)
        self.transform = transform
        self.samples: list[tuple[Path, int]] = []
        exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

        for label_idx, class_name in enumerate(CLASS_NAMES):
            class_dir = self.root / class_name
            if not class_dir.is_dir():
                continue
            for path in sorted(class_dir.rglob("*")):
                if path.suffix.lower() in exts:
                    self.samples.append((path, label_idx))

        if not self.samples:
            raise FileNotFoundError(
                f"No images found under {self.root}. "
                f"Expected subfolders: {CLASS_NAMES}"
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


print("Transforms ready.")

## 3. Train / val / test split & dataloaders

In [ ]:
def build_loaders():
    train_dir = DATA_DIR / "train"
    val_dir = DATA_DIR / "val"
    test_dir = DATA_DIR / "test"

    if not train_dir.exists():
        raise FileNotFoundError(
            f"Missing {train_dir}. Prepare data or run model/prepare_demo_data.py"
        )

    # Full train folder (labels only) — we may split off validation
    train_base = ChestXrayFolder(train_dir, transform=None)

    if val_dir.exists() and any(val_dir.iterdir()):
        train_ds = ChestXrayFolder(train_dir, transform=train_transform)
        val_ds = ChestXrayFolder(val_dir, transform=eval_transform)
        print(f"Using existing val/ folder ({len(val_ds)} images)")
    else:
        n_total = len(train_base)
        n_val = max(1, int(n_total * VAL_RATIO))
        n_train = n_total - n_val
        gen = torch.Generator().manual_seed(SEED)
        train_idx, val_idx = random_split(
            range(n_total), [n_train, n_val], generator=gen
        )
        train_idx = list(train_idx)
        val_idx = list(val_idx)

        class TransformSubset(Dataset):
            def __init__(self, base: ChestXrayFolder, indices, transform):
                self.base = base
                self.indices = list(indices)
                self.transform = transform

            def __len__(self):
                return len(self.indices)

            def __getitem__(self, i):
                path, label = self.base.samples[self.indices[i]]
                image = Image.open(path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                return image, label

        train_ds = TransformSubset(train_base, train_idx, train_transform)
        val_ds = TransformSubset(train_base, val_idx, eval_transform)
        print(
            f"No val/ folder — split train into "
            f"{len(train_ds)} train / {len(val_ds)} val (ratio={VAL_RATIO})"
        )

    if test_dir.exists() and any((test_dir / c).exists() for c in CLASS_NAMES):
        test_ds = ChestXrayFolder(test_dir, transform=eval_transform)
    else:
        # Fall back: evaluate on validation split
        test_ds = val_ds
        print("Warning: no test/ folder — using validation set for final metrics")

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )

    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders()

## 4. Model — ResNet18 (ImageNet pretrained)

In [ ]:
def build_resnet18(num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    except Exception:
        model = models.resnet18(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


model = build_resnet18(num_classes=len(CLASS_NAMES), pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)
print(model.fc)

## 5. Training loop with checkpointing

In [ ]:
def run_epoch(model, loader, criterion, optimizer, train: bool):
    model.train(train)
    total_loss, total_correct, n = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device)
            labels = labels.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            preds = logits.argmax(dim=1)
            bs = images.size(0)
            total_loss += loss.item() * bs
            total_correct += (preds == labels).sum().item()
            n += bs
    return total_loss / max(n, 1), total_correct / max(n, 1)


history = []
best_val_acc = -1.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, False)
    scheduler.step(val_acc)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "seconds": round(time.time() - t0, 2),
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"{row['seconds']}s"
    )

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": CLASS_NAMES,
                "val_acc": val_acc,
                "epoch": epoch,
                "image_size": IMAGE_SIZE,
                "mean": IMAGENET_MEAN,
                "std": IMAGENET_STD,
            },
            BEST_MODEL_PATH,
        )
        print(f"  ✓ saved best checkpoint → {BEST_MODEL_PATH} (val_acc={val_acc:.4f})")

print(f"\nTraining done. Best val_acc={best_val_acc:.4f}")

## 6. Evaluation — accuracy, precision, recall, F1, AUC

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(preds.tolist())
        # Pneumonia = positive class (index 1)
        y_prob.extend(probs[:, 1].tolist())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(
            precision_score(y_true, y_pred, average="binary", zero_division=0)
        ),
        "recall": float(recall_score(y_true, y_pred, average="binary", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="binary", zero_division=0)),
        "n_samples": int(len(y_true)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "classification_report": classification_report(
            y_true, y_pred, target_names=CLASS_NAMES, zero_division=0, output_dict=True
        ),
    }
    try:
        metrics["auc"] = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        metrics["auc"] = None

    return metrics


# Load best checkpoint for final test evaluation
ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
test_metrics = evaluate(model, test_loader)

print("=== Test set metrics ===")
print(f"Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall   : {test_metrics['recall']:.4f}")
print(f"F1       : {test_metrics['f1']:.4f}")
print(f"AUC      : {test_metrics['auc']}")
print("Confusion matrix [rows=true Normal/Pneumonia, cols=pred]:")
print(np.array(test_metrics["confusion_matrix"]))

payload = {
    "best_model_path": str(BEST_MODEL_PATH),
    "best_val_acc": best_val_acc,
    "history": history,
    "test_metrics": test_metrics,
}
METRICS_PATH.write_text(json.dumps(payload, indent=2))
print(f"\nWrote metrics → {METRICS_PATH}")
print(f"Best model saved → {BEST_MODEL_PATH}")

## 7. Quick inference sanity check

In [ ]:
sample_dirs = [
    DATA_DIR / "test" / "Pneumonia",
    DATA_DIR / "test" / "Normal",
    DATA_DIR / "samples" / "Pneumonia",
    DATA_DIR / "samples" / "Normal",
]
sample_path = None
for d in sample_dirs:
    if d.exists():
        files = list(d.glob("*.*"))
        if files:
            sample_path = files[0]
            break

if sample_path is None:
    print("No sample image found for sanity check.")
else:
    model.eval()
    img = Image.open(sample_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    pred_idx = int(probs.argmax())
    print(f"Sample: {sample_path.name}")
    print(f"Prediction: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.2%})")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name}: {probs[i]:.4f}")